# Encapsulation Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. No guard rails.** Wide-open attributes accept anything — the object ends up carrying corrupt state with zero complaints.

In [ ]:
class NaiveAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance


acc = NaiveAccount("Sarah", 5000)

acc.balance = -999_999           # nonsense accepted
acc.balance = "one million"      # wrong TYPE accepted too
print(acc.balance)
# The invariant "a balance is a number, never negative"
# was broken twice and nobody was told.

**2. Handle with care.** `_name` is a convention flag: internal to the implementation, not part of the promised API.

In [ ]:
class Report:
    def __init__(self, title, rows):
        self.title = title             # public: part of the API
        self.rows = rows               # public
        self._row_count = len(rows)    # internal: derived detail
        self._cache = {}               # internal: machinery


r = Report("Q3 Sales", [["a", 1], ["b", 2]])

print(r.title)         # encouraged
print(r._row_count)    # POSSIBLE - but the underscore warned you

**3. The double-underscore rename.** Mangling renames `__pin` to `_Vault__pin` — collision avoidance with a rude escape hatch, not security.

In [ ]:
class Vault:
    def __init__(self, pin):
        self.__pin = pin               # double underscore -> mangling


v = Vault(4321)

try:
    print(v.__pin)
except AttributeError as err:
    print("AttributeError:", err)

print(v._Vault__pin)     # mangled name still reachable - clearly rude though
print(vars(v))           # the REAL stored key
# Mangling = a compile-time rename to avoid clashes, NOT secrecy.

## Part 2 — Practice

**4. Property gatekeeper.** Assignment looks unchanged to callers, but every route now passes one validating gate.

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance          # routes through the SETTER below

    @property
    def balance(self):                  # GETTER: acc.balance
        return self._balance

    @balance.setter
    def balance(self, value):           # SETTER: acc.balance = x
        if not isinstance(value, (int, float)):
            raise TypeError("balance must be a number")
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value           # NOTE: underscore slot, NOT self.balance


acc = Account("Sarah", 100)
print(acc.balance)

acc.balance = 250
print(acc.balance)

for bad in (-5, "lots"):
    try:
        acc.balance = bad
    except (TypeError, ValueError) as err:
        print(f"{err.__class__.__name__}: {err}")

**5. Read-only, computed.** Getter-only properties derive answers on demand — and refuse to be overwritten.

In [ ]:
class Employee:
    def __init__(self, name, annual_salary):
        self.name = name
        self.annual_salary = annual_salary

    @property
    def monthly_pay(self):              # getter only => read-only + computed
        return round(self.annual_salary / 12, 2)


e = Employee("Amina", 720_000)
print(e.monthly_pay)

try:
    e.monthly_pay = 10_000              # no setter exists
except AttributeError as err:
    print("AttributeError:", err)

**6. Two-way thermometer.** A computed property can be writable too: setting the derived value updates the source of truth.

In [ ]:
class Thermometer:
    def __init__(self, celsius=0.0):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return round(self.celsius * 9 / 5 + 32, 1)

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = round((value - 32) * 5 / 9, 1)


t = Thermometer(37.0)
print(t.fahrenheit)          # 98.6

t.fahrenheit = 212.0         # setting the DERIVED value...
print(t.celsius)             # ...updates the source of truth

**7. The recursive setter trap.** Writing `self.value` inside its own setter re-triggers that setter forever — validated data belongs in the `_underscore` slot.

In [ ]:
class Broken:
    def __init__(self):
        self.value = 1             # enters the setter...

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new):
        self.value = new           # ...which assigns self.value AGAIN - forever


try:
    Broken()
except RecursionError:
    print("RecursionError: the setter kept calling itself")


class Fixed:
    def __init__(self):
        self.value = 1

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new):
        self._value = new          # store in the underscore slot - done


f = Fixed()
f.value = 42
print(f.value)

## Part 3 — Challenge

**8. Mangling's real job.** Each class mangles to its OWN prefix, so parent and child can both use `__token` without colliding.

In [ ]:
class BaseForm:
    def __init__(self):
        self.__token = "base-secret"       # becomes _BaseForm__token


class ChildForm(BaseForm):
    def __init__(self):
        super().__init__()
        self.__token = "child-secret"      # becomes _ChildForm__token


form = ChildForm()
print(form._BaseForm__token)     # parent kept its own copy
print(form._ChildForm__token)    # child wrote a separate one
# Distinct mangled prefixes -> two coexisting attributes, no accidental clash.

**9. The finished bank.** Every route into the money passes one guarded gate — the invariant holds no matter who calls.

In [ ]:
class BankAccount:
    """Validated state, guarded internals, audit trail."""

    BANK_CODE = "PNB-08"                        # class constant

    def __init__(self, owner, opening_balance=0):
        self.owner = owner                      # public on purpose
        self._transactions = []                 # internal ledger
        self.balance = opening_balance          # via validating setter

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError("balance must be numeric")
        if value < 0:
            raise ValueError("balance cannot go negative")
        self._balance = value

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount                  # through the validating setter
        self._log("deposit", amount)

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError(f"cannot withdraw {amount}; only {self.balance} available")
        self.balance -= amount
        self._log("withdraw", amount)

    def _log(self, kind, amount):               # internal helper
        self._transactions.append((kind, amount))

    def statement(self):
        lines = [f"Statement {self.BANK_CODE} | {self.owner}"]
        for kind, amount in self._transactions:
            sign = "+" if kind == "deposit" else "-"
            lines.append(f"  {kind:<8} {sign}{amount}")
        lines.append(f"  closing balance: {self.balance}")
        return "\n".join(lines)


acc = BankAccount("Sarah", 1000)
acc.deposit(2500)
acc.withdraw(700)
acc.deposit(120)
print(acc.statement())

for attempt in (10_000, -3):
    try:
        acc.withdraw(attempt)
    except ValueError as err:
        print("blocked:", err)
print("final balance:", acc.balance)            # invariant intact: never negative